# 3d. Prédiction latente sans génération : DM-JEPA 1.7, un « système 1 » mesuré

**Navigation** : [<< Précédent](03c_Constrained_Decoding_Python.ipynb) | [Index](README.md) | [Suivant >>](04_Function_Calling.ipynb)

Le carnet 3b a posé le contrat des décisions typées — décider **sans générer** — et l'a fait tourner sur des familles d'encodeurs et de LLM gelés. Ce carnet ajoute la famille que sa fiche appelle *JEPA* : un moteur **non-autorégressif** qui lit un état et rend une probabilité par option, sans jamais produire de texte.

La promesse de cette famille est double : « système 1 » rapide et non génératif, pour 0,3 Md de paramètres. Ce carnet ne répète ni l'une ni l'autre : il **exécute le harnais de validation local** ([`scripts/genai-stack/dm_jepa_local_check.py`](../../../scripts/genai-stack/dm_jepa_local_check.py), issue #21997) et montre ce que le moteur fait **mesuré chez nous** — le résultat positif, la limite, et la baseline du hasard à côté.

Les chiffres de référence sont publiés dans [docs/genai/dm-jepa-local-validation.md](../../../docs/genai/dm-jepa-local-validation.md) ; ce carnet les **reproduit en direct** sur la machine qui l'exécute.

## 1. Le moteur, le harnais, et ce que la mesure veut dire

**Le contrat du moteur** : un état (une phrase de contexte), une question typée `choice` avec `K` options, et en sortie **une probabilité par option** — pas de token généré, pas de chaîne de raisonnement. C'est le contrat `choice` du carnet 3b, servi par un moteur entraîné à prédire dans l'espace latent.

**Le harnais** (issue #21997) construit 60 problèmes déterministes (graine 42) : une ancre (*chien*, *chat*, *oiseau*, *tracteur*, *klaxon*), une option correcte « On entend un {ancre} provenant de la cour », et trois familles adverses :

| Famille adverse | Exemple | Ce qu'elle teste |
|---|---|---|
| Négation | « On n'entend **aucun** {ancre} dans la cour » | La contradiction est-elle pénalisée ? |
| Distracteur partageant l'ancre | « Le {ancre} du voisin est visible depuis la fenêtre » | Le modèle apparie-t-il le **mot**, ou le **sens** ? |
| Neutres | « Le bilan trimestriel est positif » | Plancher de référence |

**La baseline** : un tri aléatoire place l'option correcte au rang moyen `(K+1)/2` — 3,5 à six options. Une mesure ne dit quelque chose que si elle bat ce plancher.

Deux artefacts notoires, documentés dans la page de validation : le dépôt HF ne se charge pas tel quel (le paquet `dm_bert` n'est pas publié — le harnais installe un alias `sys.modules`, et le fichier `dm_jepa_1_7_engine.py` à la racine n'est pas couvert par les motifs de téléchargement du harnais : la cellule suivante le complète par l'API officielle du hub) ; et les benchmarks de la fiche (GSM8K…) ne sont pas publiques, donc **aucun chiffre de la fiche n'est validé ici**.

In [1]:
# Execution du harnais officiel (issue #21997) : aucune reimplementation.
# Prerequis : torch, tokenizers, safetensors, huggingface_hub, et le kit
# apolinario/decision-index (DECISION_INDEX_KIT=<chemin du clone>).
# NB : l'engine du depot HF interroge le device CUDA au chargement (bf16) meme
# pour une inference CPU -- un device doit rester visible, ou un build torch
# CPU-only. La re-execution commisee ici a ete lancee avec CUDA_VISIBLE_DEVICES
# pointant le GPU reserve de la machine, pas un GPU partage.
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'scripts' / 'genai-stack').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
HARNESS = ROOT / 'scripts' / 'genai-stack' / 'dm_jepa_local_check.py'
REVISION = '9f2f6f85a2923c0f2307cc966b7ead36723e96dc'

KIT = os.environ.get('DECISION_INDEX_KIT', str(Path.home() / 'decision-index'))
if not (Path(KIT) / 'decision_index').is_dir():
    raise SystemExit(
        'Kit apolinario/decision-index introuvable : '
        'git clone https://github.com/apolinario/decision-index, '
        f'puis DECISION_INDEX_KIT=<chemin> (cherche : {KIT})'
    )

snap_env = os.environ.get('DM_JEPA_SNAPSHOT')
if snap_env:
    snap = Path(snap_env)
else:
    from huggingface_hub import hf_hub_download, snapshot_download
    snap = Path(snapshot_download(repo_id='DangerLabs/dm-jepa-1.7', revision=REVISION,
                                  allow_patterns=['*.json', '*.safetensors', '*.pt', 'src/*.py']))
    if not (snap / 'dm_jepa_1_7_engine.py').exists():
        # Finding 1 de la page de validation : l'engine a la racine du depot HF
        # n'est pas dans les allow_patterns du harnais. Completion par l'API du hub.
        hf_hub_download('DangerLabs/dm-jepa-1.7', filename='dm_jepa_1_7_engine.py',
                        revision=REVISION)

REPORT = Path(tempfile.gettempdir()) / 'dm_jepa_demo_results.json'
cmd = [sys.executable, str(HARNESS), '--snapshot', str(snap), '--kit-path', KIT,
       '--device', 'cpu', '--json', str(REPORT)]
print(f"harnais : {HARNESS.name} --device cpu --json {REPORT.name}")
run = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
if run.returncode != 0:
    print(run.stdout[-800:])
    print(run.stderr[-1500:])
run.check_returncode()

report = json.loads(REPORT.read_text(encoding='utf-8'))
rk = report['ranking']
print(f"Revision {report['snapshot_revision'][:12]} | {report['param_count']:,} parametres | "
      f"{report['n_problems']} problemes, K={report['k_options']}, graine {report['seed']}")
print(f"Latence CPU mediane sur cette machine : {report['latency_ms_cpu']['median']} ms "
      f"(p90 {report['latency_ms_cpu']['p90']} ms)")
print()

import pandas as pd
demo_df = pd.DataFrame([
    {'option': "correcte « on entend un {ancre} »",
     'rang moyen': rk['good']['mean_rank'],
     'exactitude au rang 1': rk['good']['accuracy_at_1']},
    {'option': "négation « on n'entend aucun {ancre} »",
     'rang moyen': rk['negation']['mean_rank'],
     'exactitude au rang 1': rk['negation']['accuracy_at_1']},
    {'option': "distracteur « le {ancre} du voisin »",
     'rang moyen': rk['shared_word_distractor']['mean_rank'],
     'exactitude au rang 1': rk['shared_word_distractor']['accuracy_at_1']},
])
print(f"Baseline du tri aleatoire : rang moyen attendu {rk['random_expected_rank']}")
demo_df


<USER_PATH>\miniconda3\envs\coursia-genai-audio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3781.80it/s]

harnais : dm_jepa_local_check.py --device cpu --json dm_jepa_demo_results.json


Revision 9f2f6f85a292 | 349,030,286 parametres | 60 problemes, K=6, graine 42
Latence CPU mediane sur cette machine : 129.8 ms (p90 136.8 ms)



Baseline du tri aleatoire : rang moyen attendu 3.5


,option,rang moyen,exactitude au rang 1
0,correcte « on entend un {ancre} »,1.717,0.333
1,négation « on n'entend aucun {ancre} »,3.750,0.000
2,distracteur « le {ancre} du voisin »,5.700,0.000


### Lecture du résultat — le positif, la limite, la baseline

Trois lignes, trois enseignements différents :

1. **Le positif — le scoring discrimine.** L'option correcte sort au rang moyen **1,72** contre **3,5** pour un tri aléatoire : le moteur n'élit pas au hasard, loin de là. À six options, il place la bonne réponse en tête une fois sur trois (20/60) ; dans les 40 autres cas — le complément de cette exactitude — elle arrive à un autre rang.
2. **Le résultat le plus instructif — le distracteur qui partage l'ancre coule.** « Le chien du voisin est visible depuis la fenêtre » contient le mot de l'ancre, mais pas le sens attendu (« on **entend** … provenant de la **cour** »). Son rang moyen : **5,70 sur 6** — dernière place dans 43 cas sur 60. Le moteur ne fait pas de l'appariement lexical : il distingue *entendre depuis la cour* de *être visible à la fenêtre*.
3. **La limite mesurée — la négation n'est pas pénalisée.** « On n'entend **aucun** chien » contredit la question posée ; son rang moyen (**3,75**) est celui du hasard (3,5). Le moteur traite cette phrase comme une neutre : il n'est pas sensible à la polarité. C'est une **limite mesurée**, pas une inférence — elle interdit de présenter ce moteur comme comprenant la négation.

Ces trois nombres sont ceux de la page de validation (mesurés sur GPU) **reproduits ici sur CPU** : la construction du jeu (graine 42) et le scoring sont déterministes, les rangs ne dépendent pas du device.

## 2. Voir le moteur décider — un exemple complet, en direct

Le tableau moyen ne montre pas ce que le moteur rend sur **un** cas. La cellule suivante charge l'engine (chaque fonction vient du harnais, aucune n'est réécrite) et score le problème 0 : ses six options, la probabilité de chacune, dans l'ordre du classement.

In [2]:
# Fonctions du harnais : alias de paquet, chargement d'engine, construction du
# probleme, question typee. Aucune reimplementation.
import importlib.util

spec = importlib.util.spec_from_file_location('dm_jepa_check', HARNESS)
check = importlib.util.module_from_spec(spec)
spec.loader.exec_module(check)

check.alias_dm_bert(snap)
eng_mod = check.load_engine_module(snap, KIT)
engine = eng_mod.DMJEPA17Engine(model=check.REPO_ID, device='cpu')

problem = check.build_problem(0, seed=42, k=6)
probs = check.ask(engine, problem)
ordre = sorted(probs.items(), key=lambda kv: -kv[1])
roles = {problem['good']: 'correcte', problem['neg']: 'négation',
         problem['shared']: 'distracteur/ancre'}

print('Contexte :', problem['context'])
print('Consigne :', problem['instructions'])
print()
for rang, (cle, p) in enumerate(ordre, 1):
    idx = int(cle.replace('opt', ''))
    print(f"{rang}. p={p:.3f}  [{roles.get(idx, 'neutre'):>16s}]  {problem['options'][idx]}")

Contexte : Quel bruit vient de la cour ? (cas 0)
Consigne : Choisir l'option decrivant la source du bruit.

1. p=0.293  [          neutre]  La reunion commence a neuf heures
2. p=0.201  [        correcte]  On entend un tracteur provenant de la cour
3. p=0.145  [        négation]  On n'entend aucun tracteur dans la cour
4. p=0.144  [          neutre]  Le dossier est range dans l'armoire
5. p=0.135  [          neutre]  Le train part de la voie sept
6. p=0.081  [distracteur/ancre]  Le tracteur du voisin est visible depuis la fenetre


### Lecture de l'exemple

Sur ce cas précis, le moteur **ne met pas l'option correcte en tête** : une neutre (« La réunion commence à neuf heures ») la devance de neuf points de probabilité, et l'option correcte arrive deuxième. C'est la variance individuelle qui se cache derrière le tableau moyen — l'exactitude au rang 1 n'est que 33 % (20 des 60 problèmes), donc dans 40 cas sur 60 la bonne réponse n'arrive pas en tête. Un exemple unique, choisi au hasard, ne montre pas la moyenne ; il la rappelle.

Ce qui, en revanche, se voit déjà cas par cas : le distracteur qui partage le mot de l'ancre **ferme la marche** (p=0,081, près de quatre fois moins que la neutre de tête) — c'est la signature qui généralise (dernière place dans 43 cas sur 60). Et la négation reste noyée dans le paquet des neutres, ni récompensée ni pénalisée.

Une passe avant, six nombres, aucun token généré : c'est tout ce que le moteur fait. Et ce cas rappelle pourquoi le jugement porte sur 60 problèmes plutôt que sur un exemple — fusse-t-il bien tombé.

## 3. Exercices

**Exercice 1 — Passer à quatre options.**

La page de validation rapporte qu'à K=4 le rang moyen de l'option correcte tombe à 1,267 et celui du distracteur à 3,883 (sur 4). Relancez le harnais en K=4 et vérifiez ces deux nombres sur votre machine.

In [3]:
# Exercice 1 a completer : relancer le harnais en K=4 et extraire les deux rangs moyens.
cmd_k4 = [sys.executable, str(HARNESS), '--snapshot', str(snap), '--kit-path', KIT, '--device', 'cpu']
# TODO etudiant : completer cmd_k4 (--k-options 4 et --json vers un fichier temporaire),
# executer le sous-processus, charger le rapport JSON produit.
resultat_k4 = None  # TODO etudiant : (good.mean_rank, shared_word_distractor.mean_rank) a K=4
print(resultat_k4 if resultat_k4 is not None else 'Exercice a completer')

Exercice a completer


**Exercice 2 — Vos propres distracteurs.**

Les neutres du harnais sont sept phrases fixes (« Le bilan trimestriel est positif », …). Remplacez-en trois par des phrases de votre cru **qui partagent un mot avec la consigne** (« cour », « bruit ») et relancez la mesure de rang sur une dizaine de problèmes : un distracteur lexical non prévu fait-il couler l'option correcte ?

In [4]:
# Exercice 2 a completer : construire des problemes avec vos propres neutres.
# Indice : check.build_problem(i, seed, k) tire les neutres de check.NEUTRALS ;
# sauvegarder la liste d'origine, remplacer trois entrees, reconstruire, scorer.
mes_neutres = None  # TODO etudiant : trois phrases partageant un mot avec la consigne
rangs = None  # TODO etudiant : rang moyen de l'option correcte sur 10 problemes
print(rangs if rangs is not None else 'Exercice a completer')

Exercice a completer


**Exercice 3 — Le piège de duplication.**

Le rapport du harnais mesure qu'une option dupliquée reçoit **exactement** la même probabilité que l'originale (écart 0,0) et que le couple agrégé passe au rang 1 : dupliquer une option double la masse de son concept. Reproduisez-le sur le problème 0.

In [5]:
# Exercice 3 a completer : dupliquer l'option correcte et observer les probabilites.
# Indice : check.ask(engine, problem, extra_criteria={'opt_dup': problem['options'][problem['good']]})
ecart_jumeaux = None  # TODO etudiant : |p(opt_good) - p(opt_dup)| apres l'appel
print(ecart_jumeaux if ecart_jumeaux is not None else 'Exercice a completer')

Exercice a completer


## 4. Ce que cette démo ne montre pas

- **Aucun benchmark de la fiche n'est validé** : les charges Decision Index 0.3 ne sont pas publiques (401 documenté) ; les scores GSM8K et consorts ne sont ni confirmés ni infirmés.
- **La latence revendiquée (34,3 ms) n'est pas reproduite** : la page de validation mesure 56,7 à 89,7 ms sur GPU partagé. La latence CPU affichée plus haut dépend de la machine — elle sert à dimensionner, pas à comparer à la fiche.
- **La négation n'est pas comprise** (rang du hasard) : ce moteur trie des options, il ne raisonne pas sur la polarité.
- **Le contrat est fermé** : comme tout `choice`, il ne dit rien hors de l'ensemble d'options proposé — et la duplication d'option le piège (exercice 3).

## Conclusion

Un moteur de décision non génératif **peut** battre largement le hasard sur un contrat fermé — et le faire en une passe, sans générer un token. La mesure locale montre mieux qu'une fiche : le distracteur lexical coule (le moteur lit le sens, pas le mot), mais la négation passe inaperçue (il ne lit pas la polarité). C'est la division du travail que le carnet 3b cherchait : le « système 1 » décide vite sur un espace fermé, et **tout ce qui exige de raisonner sur la question reste au « système 2 » génératif**.

**Références** : harnais et plan de validation — issue #21997 ; page de mesures — [docs/genai/dm-jepa-local-validation.md](../../../docs/genai/dm-jepa-local-validation.md) ; issue de la démo — #22246 ; carnet précédent — `03b_Typed_Decisions_System1.ipynb`.

***

**Navigation** : [<< Précédent](03c_Constrained_Decoding_Python.ipynb) | [Index](README.md) | [Suivant >>](04_Function_Calling.ipynb)
